# COF Vision · entrenamiento de **RF-DETR** (personas y cabezas) para el conteo en buses

**Qué hace:** arma un conjunto de datos con tus videos de buses (CH4, CH5 y CH7 de PGBY-49, PGTV-28 y SKWD-57: día, noche e infrarrojo, con prioridad a las aglomeraciones), más las imágenes con cabezas marcadas por Qwen. Después entrena **RF-DETR Nano**, el detector que ganó la prueba contra YOLO, D-FINE y RT-DETR, para detectar **personas (clase 0)** y **cabezas (clase 1)**.

- Las personas las marca un «profesor» grande, **RF-DETR Large**.
- El aumento de datos está hecho para cámaras de bus: sol, noche, blanco y negro como el infrarrojo, compresión del DVR y desenfoque.
- Las paradas de SKWD-57 que conté a mano quedan **fuera** del entrenamiento, para medir el modelo después sin trampa.

**Antes de empezar:**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** → Guardar.
2. En tu Drive deben estar las carpetas **cof_datos** (las 5 partes) y **cof_modelos** (con `cof-heads-s-colab.pt`).
3. Menú **Entorno de ejecución → Ejecutar todas** y acepta el permiso de Google Drive.

**Duración:** armar los datos ~40–60 min; entrenar ~2–3 h. Si Colab se corta, vuelve a abrir este cuaderno y aprieta **Ejecutar todas**: sigue donde quedó, porque todo se guarda en Drive en la carpeta **cof_rfdetr**.

**Al final** queda en **Mi unidad/cof_rfdetr/modelos** el archivo `cof-rfdetr-nano.onnx` (≈29 MB) y `resultados_rfdetr.txt`. Comparte esa carpeta (cualquier persona con el enlace) y envía el enlace.

In [ ]:
# 1) Tarjeta gráfica e instalación
!nvidia-smi -L || echo 'SIN GPU: cambia el entorno de ejecución a T4 GPU y vuelve a ejecutar todo'
!pip -q install "rfdetr[train,augment]==1.11.2" ultralytics==8.4.164 onnx onnxruntime

In [ ]:
# 2) Google Drive y carpetas
from google.colab import drive
drive.mount('/content/drive')
import os, glob
D = '/content/drive/MyDrive'
DATOS = f'{D}/cof_datos'
SALIDA = f'{D}/cof_rfdetr'
os.makedirs(SALIDA, exist_ok=True)
assert glob.glob(f'{DATOS}/*parte*.zip'), 'No encuentro las partes en Mi unidad/cof_datos'
PROFE = next((p for p in [f'{D}/cof_modelos/cof-heads-s-colab.pt', f'{D}/cof_modelos/cof-heads-n-colab.pt'] if os.path.exists(p)), None)
assert PROFE, 'No encuentro el detector de cabezas en Mi unidad/cof_modelos (cof-heads-s-colab.pt)'
print('Profesor de cabezas:', PROFE)

In [ ]:
%%writefile /content/cof_rf_datos.py
# COF Vision · datos para entrenar RF-DETR (personas y cabezas) con los videos de los buses.
#
# Clases: 0 = persona, 1 = cabeza.
#  - Cabezas de las imágenes marcadas por Qwen (cof_datos): se usan tal cual (son la referencia).
#  - Personas de esas imágenes y personas + cabezas de los fotogramas nuevos de los videos: las marcan dos
#    «profesores» grandes (RF-DETR Large para personas y el detector de cabezas entrenado en Colab).
#  - Toda iluminación: día, noche e infrarrojo. Además se agregan copias en blanco y negro y con poca luz.
#  - Aglomeraciones: los fotogramas con mucha gente se repiten en el entrenamiento.
#  - La prueba (val) son solo imágenes con cabezas marcadas por Qwen, de momentos que no entran al entrenamiento.
import argparse, glob, json, os, random, shutil, subprocess, sys

import cv2
import numpy as np

# Cámaras de puerta (CH4 delantera, CH5 central, CH7 trasera) de las carpetas de video compartidas.
VIDEOS = {
    'pgby49_CH4': '1DmD3CCz3h2hD-pXgc2MLHlXcxBg9tSGR',
    'pgby49_CH5': '1AMeTGUAIeeP9KpCUieN0fH9d0XnhIqXW',
    'pgby49_CH7': '1nzxLljdXSlR8-yEo2YfEZiwJZWhs3NLT',
    'pgtv28_CH4': '11a6WgSglwKdsePWWTprA-C3nKw_q78Jx',
    'pgtv28_CH5': '15syhEZ8NL0ACrZiEzZEPl0q80UELQ8ni',
    'pgtv28_CH7': '12wxYnzy-hTCR_4Tj6d8f-VVB3_u-t5MJ',
    'skwd57_CH4': '11UBtPbT_AHuVKtUiYi1p0aHFHq4oFA7-',
    'skwd57_CH5': '19AJyyjWtNzUe1prYGM1JMii0S8l8oW-r',
    'skwd57_CH7': '1m763Lfj1PwI8Gu9rjO6nQpzzjc2yAR0d',
}
# Momentos que se dejan fuera del entrenamiento (segundos del video): la prueba (val) de PGTV-28 y las paradas de
# SKWD-57 contadas a mano, con las que después se compara el conteo.
VAL_WINDOWS = {'pgtv28': [(3650, 4250)], 'skwd57': [(185, 490), (660, 700), (1175, 1205)]}
NAMES = {0: 'persona', 1: 'cabeza'}


def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


def write_labels(path, rows):
    with open(path, 'w') as f:
        for c, x, y, w, h in rows:
            f.write(f'{c} {x:.5f} {y:.5f} {w:.5f} {h:.5f}\n')


def read_labels(path):
    if not os.path.exists(path):
        return []
    out = []
    for line in open(path):
        p = line.split()
        if len(p) == 5:
            out.append((int(p[0]), *map(float, p[1:])))
    return out


def boxes(model, imgs, cls, conf, imgsz, augment=False):
    """Recuadros (x, y, w, h normalizados) de una clase en un lote de imágenes."""
    res = model.predict(imgs, imgsz=imgsz, conf=conf, iou=0.6, max_det=400, classes=[cls] if cls is not None else None,
                        augment=augment, verbose=False)
    out = []
    for r in res:
        b = r.boxes.xywhn.cpu().numpy() if len(r.boxes) else np.zeros((0, 4))
        c = r.boxes.conf.cpu().numpy() if len(r.boxes) else np.zeros((0,))
        out.append([(tuple(map(float, v)), float(k)) for v, k in zip(b, c) if v[2] * v[3] > 0.0004])
    return out


class RFProfe:
    """RF-DETR Large (COCO) como profesor de personas, con la misma salida que boxes() de YOLO."""
    def __init__(self):
        import rfdetr
        self.m = rfdetr.RFDETRLarge()
        try:
            self.m.optimize_for_inference()
        except Exception as e:
            print('RF-DETR sin optimizar:', e)

    def boxes(self, imgs, conf):
        from PIL import Image
        ims = [Image.open(p).convert('RGB') if isinstance(p, str) else Image.fromarray(cv2.cvtColor(p, cv2.COLOR_BGR2RGB)) for p in imgs]
        out = []
        for k in range(0, len(ims), 8):
            ds = self.m.predict(ims[k:k + 8], threshold=conf)
            ds = ds if isinstance(ds, list) else [ds]
            for im, d in zip(ims[k:k + 8], ds):
                W, H = im.size
                r = []
                for (x0, y0, x1, y1), c, cl in zip(d.xyxy, d.confidence, d.class_id):
                    if int(cl) != 1:  # COCO: 1 = persona
                        continue
                    w, h = (x1 - x0) / W, (y1 - y0) / H
                    if w * h > 0.0004:
                        r.append((((x0 + x1) / 2 / W, (y0 + y1) / 2 / H, w, h), float(c)))
                out.append(r)
        return out


def only(bs):
    return [b for b, _ in bs]


def iou(a, b):
    ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw, ih = max(0, min(ax1, bx1) - max(ax0, bx0)), max(0, min(ay1, by1) - max(ay0, by0))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9)


def clean_heads(qwen, teacher, teacher_conf, persons):
    """Control cruzado: una cabeza de Qwen se queda si el detector de cabezas también la ve o si cae en la parte de
    arriba de una persona (así se van las marcas sobre el texto de la pantalla o reflejos). Las cabezas que Qwen no
    marcó y el detector ve con confianza alta (≥0.55) se agregan: en las multitudes Qwen se salta algunas."""
    keep = []
    for h in qwen:
        in_person = any(abs(h[0] - p[0]) < p[2] / 2 and p[1] - p[3] / 2 - 0.02 <= h[1] <= p[1] for p in persons)
        if in_person or any(iou(h, t) >= 0.3 for t in teacher):
            keep.append(h)
    for t, c in zip(teacher, teacher_conf):
        if c >= 0.55 and not any(iou(t, h) >= 0.3 for h in keep):
            keep.append(t)
    return keep


def is_mono(img):
    b, g, r = cv2.split(img.astype(np.int16))
    return float(np.mean(np.abs(b - g)) + np.mean(np.abs(g - r))) < 6


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--datos', required=True, help='carpeta con cof_cabezas_parte*.zip')
    ap.add_argument('--trabajo', required=True, help='carpeta local de trabajo')
    ap.add_argument('--cabezas_profe', required=True, help='modelo de cabezas para marcar los fotogramas nuevos (.pt)')
    ap.add_argument('--videos', default='/content/videos')
    ap.add_argument('--cada', type=float, default=1.5, help='segundos entre fotogramas candidatos')
    ap.add_argument('--por_video', type=int, default=650, help='fotogramas nuevos por video')
    ap.add_argument('--test', action='store_true', help='prueba rápida (pocas imágenes)')
    ap.add_argument('--solo', default='', help='solo estos videos (separados por coma), para pruebas')
    a = ap.parse_args()
    from ultralytics import YOLO
    T = a.trabajo
    ds = f'{T}/cof_personas_cabezas'
    if os.path.exists(f'{ds}/listo.txt'):
        print('El conjunto de datos ya estaba armado:', ds)
        return
    shutil.rmtree(ds, ignore_errors=True)
    for s in ('train', 'val'):
        os.makedirs(f'{ds}/images/{s}', exist_ok=True)
        os.makedirs(f'{ds}/labels/{s}', exist_ok=True)

    # 1) Imágenes con cabezas marcadas por Qwen.
    base = f'{T}/base'
    if not os.path.exists(f'{base}/cof_cabezas/data.yaml'):
        os.makedirs(base, exist_ok=True)
        for z in sorted(glob.glob(f'{a.datos}/*parte*.zip')):
            sh(f'unzip -q -o "{z}" -d "{base}"')
    profe_p = RFProfe()
    profe_c = YOLO(a.cabezas_profe)
    stats = {'base': 0, 'video': 0, 'gris': 0, 'oscura': 0, 'multitud_extra': 0}
    for s in ('train', 'val'):
        ims = sorted(glob.glob(f'{base}/cof_cabezas/images/{s}/*.jpg'))
        if a.test:
            ims = ims[:24]
        for i in range(0, len(ims), 16):
            chunk = ims[i:i + 16]
            pers = profe_p.boxes(chunk, 0.40)
            tch = boxes(profe_c, chunk, None, 0.25, 960, augment=True)
            for p, pb, th in zip(chunk, pers, tch):
                n = os.path.splitext(os.path.basename(p))[0]
                qwen = [tuple(r[1:]) for r in read_labels(f'{base}/cof_cabezas/labels/{s}/{n}.txt')]
                heads = clean_heads(qwen, only(th), [k for _, k in th], only(pb))
                stats['cabezas_qwen'] = stats.get('cabezas_qwen', 0) + len(qwen)
                stats['cabezas_finales'] = stats.get('cabezas_finales', 0) + len(heads)
                shutil.copy(p, f'{ds}/images/{s}/{n}.jpg')
                write_labels(f'{ds}/labels/{s}/{n}.txt', [(0, *b) for b in only(pb)] + [(1, *h) for h in heads])
                stats['base'] += 1
        print(s, len(ims), 'imágenes base con personas agregadas', flush=True)

    # 2) Fotogramas nuevos de los videos (toda la grabación: día, noche, infrarrojo), sobre todo con mucha gente.
    os.makedirs(a.videos, exist_ok=True)
    rng = random.Random(7)
    for name, fid in VIDEOS.items():
        if a.solo and name not in a.solo.split(','):
            continue
        mp4 = f'{a.videos}/{name}.mp4'
        if not os.path.exists(mp4) or os.path.getsize(mp4) < 1e6:
            sh(f'curl -sL -o "{mp4}" "https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t"')
        cap = cv2.VideoCapture(mp4)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        dur = (cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0) / fps
        if a.test:
            dur = min(dur, 240)
        bus = name.split('_')[0]
        wins = VAL_WINDOWS.get(bus, [])
        cand = []
        t = 0.0
        while t < dur:
            if not any(w0 <= t <= w1 for w0, w1 in wins):
                cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
                ok, f = cap.read()
                if ok:
                    cand.append((t, cv2.resize(f, (640, round(640 * f.shape[0] / f.shape[1])))))
            t += a.cada
            # Se cuentan personas por tandas para no llenar la memoria.
            if len(cand) >= 256 or t >= dur:
                cnt = [len(b) for b in profe_p.boxes([c[1] for c in cand], 0.45)] if cand else []
                for (tt, f), k in zip(cand, cnt):
                    if k >= 6 or (k >= 2 and rng.random() < 0.35) or (k == 0 and rng.random() < 0.02):
                        cv2.imwrite(f'{a.videos}/{name}_{int(tt * 10):06d}_{k:02d}.jpg', f, [cv2.IMWRITE_JPEG_QUALITY, 90])
                cand = []
        sel = sorted(glob.glob(f'{a.videos}/{name}_*.jpg'))
        # Primero las multitudes, después el resto, hasta el tope por video.
        sel.sort(key=lambda p: -int(p[-6:-4]))
        sel = sel[:(40 if a.test else a.por_video)]
        for i in range(0, len(sel), 16):
            chunk = sel[i:i + 16]
            pers = profe_p.boxes(chunk, 0.40)
            heads = boxes(profe_c, chunk, None, 0.30, 960, augment=True)
            for p, pb, hb in zip(chunk, pers, heads):
                n = os.path.splitext(os.path.basename(p))[0]
                shutil.copy(p, f'{ds}/images/train/{n}.jpg')
                write_labels(f'{ds}/labels/train/{n}.txt', [(0, *b) for b in only(pb)] + [(1, *b) for b in only(hb)])
                stats['video'] += 1
        print(name, len(sel), 'fotogramas nuevos', flush=True)

    # 3) Iluminación: copias en blanco y negro (como el infrarrojo) y con poca luz; multitudes repetidas.
    rng = random.Random(11)
    for p in sorted(glob.glob(f'{ds}/images/train/*.jpg')):
        n = os.path.splitext(os.path.basename(p))[0]
        lab = read_labels(f'{ds}/labels/train/{n}.txt')
        img = cv2.imread(p)
        if img is None:
            continue
        if not is_mono(img) and rng.random() < 0.30:
            g = cv2.cvtColor(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
            cv2.imwrite(f'{ds}/images/train/{n}_gris.jpg', g, [cv2.IMWRITE_JPEG_QUALITY, 90])
            write_labels(f'{ds}/labels/train/{n}_gris.txt', lab)
            stats['gris'] += 1
        if rng.random() < 0.15:
            d = np.clip(255 * (img / 255.0) ** 1.8 * 0.7, 0, 255).astype(np.uint8)
            cv2.imwrite(f'{ds}/images/train/{n}_oscura.jpg', d, [cv2.IMWRITE_JPEG_QUALITY, 90])
            write_labels(f'{ds}/labels/train/{n}_oscura.txt', lab)
            stats['oscura'] += 1
        if sum(1 for r in lab if r[0] == 0) >= 8 or sum(1 for r in lab if r[0] == 1) >= 8:
            shutil.copy(p, f'{ds}/images/train/{n}_multitud.jpg')
            write_labels(f'{ds}/labels/train/{n}_multitud.txt', lab)
            stats['multitud_extra'] += 1
    with open(f'{ds}/data.yaml', 'w') as f:
        f.write(f'path: {ds}\ntrain: images/train\nval: images/val\nnames:\n  0: persona\n  1: cabeza\n')
    stats['train_total'] = len(glob.glob(f'{ds}/images/train/*.jpg'))
    stats['val_total'] = len(glob.glob(f'{ds}/images/val/*.jpg'))
    json.dump(stats, open(f'{ds}/listo.txt', 'w'))
    print('Conjunto de datos listo:', stats, flush=True)


if __name__ == '__main__':
    sys.exit(main())


In [ ]:
%%writefile /content/cof_rf_entrenar.py
# COF Vision · entrena RF-DETR (personas y cabezas) con los datos de los buses, guarda cada avance en Drive para
# seguir si Colab se corta, y exporta el modelo para la página (ONNX, misma entrada y salida que usa la página).
import argparse, glob, json, os, shutil, sys
import torch, torch.nn as nn, torch.nn.functional as F

ap = argparse.ArgumentParser()
ap.add_argument('--data', required=True, help='carpeta del conjunto de datos (con data.yaml)')
ap.add_argument('--salida', required=True, help='carpeta en Drive')
ap.add_argument('--tam', default='nano', choices=['nano', 'small', 'medium'])
ap.add_argument('--epocas', type=int, default=80)
ap.add_argument('--lote', type=int, default=8)
ap.add_argument('--paciencia', type=int, default=15)
ap.add_argument('--device', default='cuda')
a = ap.parse_args()
import rfdetr

# Aumento de datos para cámaras de techo en buses: cambios fuertes de luz (sol, noche), blanco y negro como el
# infrarrojo, compresión del DVR, desenfoque y ruido. Sin giros grandes ni volteo vertical (la cámara está fija).
AUG_BUS = {
    'HorizontalFlip': {'p': 0.5},
    'Affine': {'scale': (0.85, 1.15), 'translate_percent': (-0.05, 0.05), 'rotate': (-5, 5), 'p': 0.4},
    'RandomBrightnessContrast': {'brightness_limit': 0.35, 'contrast_limit': 0.3, 'p': 0.6},
    'ColorJitter': {'brightness': 0.2, 'contrast': 0.2, 'saturation': 0.4, 'hue': 0.05, 'p': 0.4},
    'ToGray': {'p': 0.15},
    'GaussianBlur': {'blur_limit': 3, 'p': 0.2},
    'GaussNoise': {'std_range': (0.01, 0.05), 'p': 0.2},
    'ImageCompression': {'quality_range': (40, 90), 'p': 0.3},
}
Cls = {'nano': rfdetr.RFDETRNano, 'small': rfdetr.RFDETRSmall, 'medium': rfdetr.RFDETRMedium}[a.tam]
out = f'{a.salida}/rfdetr_{a.tam}'
os.makedirs(out, exist_ok=True)
if not os.path.exists(f'{out}/terminado.txt'):
    last = f'{out}/last.ckpt'
    print('== sigue donde quedó' if os.path.exists(last) else '== entrenamiento nuevo', flush=True)
    m = Cls()
    m.train(dataset_dir=a.data, epochs=a.epocas, batch_size=a.lote, grad_accum_steps=max(1, 16 // a.lote), lr=1e-4,
            output_dir=out, aug_config=AUG_BUS, early_stopping=True, early_stopping_patience=a.paciencia,
            checkpoint_interval=1, resume=last if os.path.exists(last) else None, device=a.device, num_workers=2,
            progress_bar='tqdm')
    open(f'{out}/terminado.txt', 'w').write('ok')

# Exportar el mejor: entrada [1,3,640,640] RGB 0-1 (relleno gris), salida [1,300,6] = x0,y0,x1,y1 (px), confianza,
# clase (0 persona, 1 cabeza). Se guarda completo y en versión liviana (int8) para la página.
best = f'{out}/checkpoint_best_total.pth'
m = Cls(pretrain_weights=best)
core = m.model.model.eval().cpu()
res = m.model.resolution
core.export()


class Wrap(nn.Module):
    def __init__(s, core, res, ncls):
        super().__init__()
        s.core, s.res = core, res
        s.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        s.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        lut = torch.full((ncls,), 3.0)
        lut[0], lut[1] = 0.0, 1.0
        s.register_buffer('lut', lut)

    def forward(s, x):
        x = F.interpolate(x, size=(s.res, s.res), mode='bilinear', align_corners=False)
        boxes, logits = s.core((x - s.mean) / s.std)[:2]
        sc, ci = logits.sigmoid().max(-1)
        cx, cy, w, h = boxes.unbind(-1)
        xyxy = torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], -1) * 640.0
        return torch.cat([xyxy, sc.unsqueeze(-1), s.lut[ci].unsqueeze(-1)], -1)


with torch.no_grad():
    ncls = core(torch.zeros(1, 3, res, res))[1].shape[-1]
W = Wrap(core, res, ncls).eval()
os.makedirs(f'{a.salida}/modelos', exist_ok=True)
f32 = f'{out}/cof-rfdetr-{a.tam}-f32.onnx'
torch.onnx.export(W, torch.rand(1, 3, 640, 640), f32, input_names=['images'], output_names=['output0'], opset_version=17,
                  do_constant_folding=True, dynamo=False)
from onnxruntime.quantization import quantize_dynamic, QuantType
q8 = f'{out}/cof-rfdetr-{a.tam}.onnx'
quantize_dynamic(f32, q8, weight_type=QuantType.QUInt8)
shutil.copy(q8, f'{a.salida}/modelos/cof-rfdetr-{a.tam}.onnx')
shutil.copy(f32, f'{a.salida}/modelos/cof-rfdetr-{a.tam}-f32.onnx')
shutil.copy(best, f'{a.salida}/modelos/cof-rfdetr-{a.tam}.pth')
# Resultados de la prueba (val) por época.
try:
    import csv
    rows = list(csv.DictReader(open(f'{out}/metrics.csv')))
    keys = [k for k in rows[-1] if 'map' in k.lower()][:6]
    best_row = max((r for r in rows if r.get(keys[0])), key=lambda r: float(r[keys[0]] or 0)) if keys else {}
    txt = f'RF-DETR {a.tam}: épocas {len(set(r.get("epoch") for r in rows))} · mejor ' + ' · '.join(f'{k} {best_row.get(k)}' for k in keys)
except Exception as e:
    txt = f'RF-DETR {a.tam}: listo (sin métricas: {e})'
open(f'{a.salida}/modelos/resultados_rfdetr.txt', 'w').write(txt + '\n')
print(txt)
print('Modelo para la página:', f'{a.salida}/modelos/cof-rfdetr-{a.tam}.onnx')


In [ ]:
# 3) Conjunto de datos (se guarda en Drive para no rehacerlo si Colab se corta)
import os
ZIP = f'{SALIDA}/cof_rfdetr_datos.zip'
if os.path.exists(ZIP):
    print('Usando el conjunto de datos ya armado en Drive')
    !unzip -q -o "{ZIP}" -d /content/trabajo
else:
    !cd /content && python cof_rf_datos.py --datos "{DATOS}" --trabajo /content/trabajo --cabezas_profe "{PROFE}" --videos /content/videos
    !cd /content/trabajo && zip -q -r -0 "{ZIP}" cof_personas_cabezas
!cat /content/trabajo/cof_personas_cabezas/listo.txt

In [ ]:
# 4) Entrenar RF-DETR Nano (personas + cabezas). Cada época queda guardada en Drive.
!cd /content && python cof_rf_entrenar.py --data /content/trabajo/cof_personas_cabezas --salida "{SALIDA}" --tam nano --epocas 80 --lote 8 --paciencia 15

In [ ]:
# 5) Resultado
!ls -la "{SALIDA}/modelos"
!cat "{SALIDA}/modelos/resultados_rfdetr.txt"
print('LISTO: comparte la carpeta cof_rfdetr/modelos (cualquier persona con el enlace) y envía el enlace.')